# 🦟 Malaria Atlas Project (MAP) - Exemplos

Este notebook demonstra como usar o accessor do Malaria Atlas Project para obter dados de malária de alta resolução espacial.

## 📋 Conteúdo

1. [Setup e Configuração](#setup)
2. [Dados de Taxa de Parasitemia (PR)](#pr-data)
3. [Limites Administrativos](#admin-boundaries)
4. [Dados de Vetores](#vector-data)
5. [Dados Raster](#raster-data)
6. [Análise de Malária no Brasil](#brazil-analysis)

## 📖 Documentação

- **MAP Website**: https://data.malariaatlas.org/
- **R Package**: https://github.com/malaria-atlas-project/malariaAtlas
- **API**: WFS/WCS (Web Feature Service / Web Coverage Service)
- **Licença**: Creative Commons Attribution 3.0

<a id='setup'></a>
## 1. Setup e Configuração

In [ ]:
import sys
sys.path.insert(0, '../../scripts/accessors')

from malaria_atlas import MalariaAtlasAccessor
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Inicializar accessor
map_accessor = MalariaAtlasAccessor()

print("✅ MalariaAtlasAccessor inicializado!")

<a id='pr-data'></a>
## 2. Dados de Taxa de Parasitemia (PR)

Dados de PR (Parasite Rate) são medições de prevalência de parasitas de malária obtidas através de inquéritos.

In [ ]:
# Listar versões disponíveis de dados PR
try:
    pr_versions = map_accessor.list_pr_versions()
    print(f"📊 Versões de dados PR disponíveis: {len(pr_versions)}")
    print(pr_versions.head())
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Buscar dados PR para o Brasil - P. falciparum
try:
    print("🔍 Buscando dados PR para Brasil (P. falciparum)...")
    pr_pf_brazil = map_accessor.get_pr_data(
        iso="BRA",
        species="Pf"
    )
    print(f"\n✅ Encontrados {len(pr_pf_brazil)} pontos de inquérito")
    
    if not pr_pf_brazil.empty:
        print("\n📊 Colunas disponíveis:")
        print(pr_pf_brazil.columns.tolist())
        
        print("\n📊 Primeiras linhas:")
        print(pr_pf_brazil.head())
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Buscar dados PR para múltiplos países da Amazônia
try:
    print("🔍 Buscando dados PR para países amazônicos...")
    
    amazon_countries = ["BRA", "COL", "PER", "BOL", "ECU"]
    pr_amazon = map_accessor.get_pr_data(
        iso=amazon_countries,
        species="Pf",
        start_date="2010-01-01",
        end_date="2020-12-31"
    )
    
    print(f"\n✅ Total de pontos na Amazônia: {len(pr_amazon)}")
    
    if not pr_amazon.empty:
        # Resumo por país
        summary = pr_amazon.groupby('country').agg({
            'pf_parasite_rate': ['count', 'mean', 'std']
        }).round(3)
        print("\n📊 Resumo por país:")
        print(summary)
        
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Visualizar distribuição de taxas de parasitemia
if not pr_amazon.empty and 'pf_parasite_rate' in pr_amazon.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histograma
    pr_amazon['pf_parasite_rate'].hist(bins=30, ax=axes[0], edgecolor='black')
    axes[0].set_xlabel('Taxa de Parasitemia (PfPR)')
    axes[0].set_ylabel('Frequência')
    axes[0].set_title('Distribuição de Taxas de Parasitemia\n(Amazônia, 2010-2020)')
    
    # Boxplot por país
    countries_order = pr_amazon.groupby('country')['pf_parasite_rate'].median().sort_values(ascending=False).index
    pr_amazon.boxplot(column='pf_parasite_rate', by='country', ax=axes[1])
    axes[1].set_xlabel('País')
    axes[1].set_ylabel('Taxa de Parasitemia (PfPR)')
    axes[1].set_title('Taxas de Parasitemia por País')
    plt.suptitle('')  # Remove título automático
    
    plt.tight_layout()
    plt.show()

<a id='admin-boundaries'></a>
## 3. Limites Administrativos

O MAP fornece limites administrativos em diferentes níveis (admin0-3) para todos os países.

In [ ]:
# Listar países disponíveis
try:
    countries = map_accessor.list_countries(admin_level="admin0")
    print(f"🌍 Total de países disponíveis: {len(countries)}")
    print("\n📊 Primeiros países:")
    print(countries.head(10))
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Buscar limites administrativos do Brasil (estados)
try:
    print("🔍 Buscando limites dos estados brasileiros...")
    brazil_states = map_accessor.get_admin_boundaries(
        iso="BRA",
        admin_level="admin1"
    )
    
    print(f"\n✅ Encontrados {len(brazil_states)} estados")
    
    if not brazil_states.empty:
        print("\n📊 Colunas disponíveis:")
        print(brazil_states.columns.tolist())
        
        print("\n📊 Primeiras linhas:")
        print(brazil_states[['iso', 'name_0', 'name_1', 'admn_level']].head())
        
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Visualizar mapa dos estados brasileiros (se geopandas disponível)
try:
    import geopandas as gpd
    
    if isinstance(brazil_states, gpd.GeoDataFrame):
        fig, ax = plt.subplots(figsize=(12, 10))
        brazil_states.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
        ax.set_title('Estados Brasileiros - Limites Administrativos MAP', fontsize=14)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ Instale geopandas para visualização de mapas")
        print("   pip install geopandas")
except ImportError:
    print("⚠️ geopandas não instalado. Instale com: pip install geopandas")
except Exception as e:
    print(f"❌ Erro: {e}")

<a id='vector-data'></a>
## 4. Dados de Vetores

Dados de ocorrência de vetores (Anopheles) para análise de distribuição espacial de mosquitos.

In [ ]:
# Listar espécies de vetores disponíveis
try:
    vector_species = map_accessor.list_vector_species()
    print(f"🦟 Total de espécies de vetores: {len(vector_species)}")
    print("\n📊 Espécies principais:")
    print(vector_species.head(15))
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Buscar ocorrências de Anopheles no Brasil
try:
    print("🔍 Buscando ocorrências de Anopheles no Brasil...")
    
    vector_occurrences = map_accessor.get_vector_occurrence(
        species=["Anopheles darlingi", "Anopheles aquasalis"],
        iso="BRA"
    )
    
    print(f"\n✅ Encontradas {len(vector_occurrences)} ocorrências")
    
    if not vector_occurrences.empty:
        print("\n📊 Colunas disponíveis:")
        print(vector_occurrences.columns.tolist())
        
        print("\n📊 Resumo por espécie:")
        print(vector_occurrences['species'].value_counts())
        
except Exception as e:
    print(f"❌ Erro: {e}")

<a id='raster-data'></a>
## 5. Dados Raster

Superfícies raster modeladas de alta resolução para burden de malária.

In [ ]:
# Listar rasters disponíveis
try:
    print("📊 Listando rasters disponíveis...")
    rasters = map_accessor.list_rasters(workspace="Malaria")
    
    print(f"\n✅ Encontrados {len(rasters)} rasters no workspace Malaria")
    
    print("\n📊 Rasters principais:")
    for idx, row in rasters.head(10).iterrows():
        print(f"   - {row['title']}")
        
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Buscar informações de um raster específico
try:
    if not rasters.empty:
        # Pegar o primeiro raster como exemplo
        dataset_id = rasters.iloc[0]['dataset_id']
        
        print(f"🔍 Obtendo informações do raster: {dataset_id}")
        
        raster_info = map_accessor.get_raster_info(dataset_id)
        
        print("\n📊 Informações do raster:")
        for key, value in raster_info.items():
            print(f"   {key}: {value}")
            
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Buscar raster de PfPR para o Brasil
try:
    # Procurar por rasters de PfPR
    pfpr_rasters = map_accessor.search_data("PfPR")
    
    if not pfpr_rasters.empty:
        print(f"\n📊 Rasters de PfPR encontrados: {len(pfpr_rasters)}")
        print(pfpr_rasters[['dataset_id', 'title']].head())
        
        # Download do primeiro raster (opcional - pode ser grande)
        # dataset_id = pfpr_rasters.iloc[0]['dataset_id']
        # raster = map_accessor.download_raster(
        #     dataset_id=dataset_id,
        #     extent=[-74, -34, -34, 5],  # Bounding box do Brasil
        #     output_path="pfpr_brazil.tif"
        # )
    else:
        print("⚠️ Nenhum raster de PfPR encontrado")
        
except Exception as e:
    print(f"❌ Erro: {e}")

<a id='brazil-analysis'></a>
## 6. Análise de Malária no Brasil

Exemplo de análise integrada usando dados do MAP.

In [ ]:
# Análise temporal de dados PR no Brasil
try:
    print("🔍 Analisando tendências temporais de malária no Brasil...")
    
    # Buscar todos os dados PR para o Brasil
    pr_brazil_all = map_accessor.get_pr_data(
        iso="BRA",
        species="Pf"
    )
    
    if not pr_brazil_all.empty and 'year_start' in pr_brazil_all.columns:
        # Converter ano para numérico
        pr_brazil_all['year'] = pd.to_numeric(pr_brazil_all['year_start'], errors='coerce')
        
        # Agrupar por ano
        yearly_data = pr_brazil_all.groupby('year').agg({
            'pf_parasite_rate': ['count', 'mean', 'std']
        }).reset_index()
        
        yearly_data.columns = ['year', 'n_surveys', 'mean_pr', 'std_pr']
        
        print("\n📊 Resumo anual:")
        print(yearly_data.dropna().tail(10))
        
        # Plotar tendência temporal
        fig, ax1 = plt.subplots(figsize=(12, 6))
        
        color = 'tab:blue'
        ax1.set_xlabel('Ano')
        ax1.set_ylabel('Número de Inquéritos', color=color)
        ax1.plot(yearly_data['year'], yearly_data['n_surveys'], color=color, marker='o')
        ax1.tick_params(axis='y', labelcolor=color)
        
        ax2 = ax1.twinx()
        color = 'tab:red'
        ax2.set_ylabel('Taxa de Parasitemia Média', color=color)
        ax2.plot(yearly_data['year'], yearly_data['mean_pr'], color=color, marker='s')
        ax2.tick_params(axis='y', labelcolor=color)
        
        plt.title('Malária no Brasil: Inquéritos e Taxas de Parasitemia', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# Mapa de distribuição de casos (se matplotlib com scatter disponível)
try:
    if not pr_brazil_all.empty and 'latitude' in pr_brazil_all.columns:
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Scatter plot com cores baseadas na taxa de parasitemia
        scatter = ax.scatter(
            pr_brazil_all['longitude'],
            pr_brazil_all['latitude'],
            c=pr_brazil_all['pf_parasite_rate'],
            cmap='YlOrRd',
            s=50,
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5
        )
        
        plt.colorbar(scatter, label='Taxa de Parasitemia (PfPR)')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_title('Distribuição Espacial de Inquéritos de Malária no Brasil', fontsize=14)
        
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"❌ Erro: {e}")

## 📚 Resumo

Este notebook demonstrou:

1. ✅ **Setup** do accessor Malaria Atlas Project
2. ✅ **Dados PR** - Taxas de parasitemia de inquéritos
3. ✅ **Limites Administrativos** - Shapefiles de países e estados
4. ✅ **Dados de Vetores** - Ocorrência de Anopheles
5. ✅ **Dados Raster** - Superfícies modeladas de malária
6. ✅ **Análise integrada** - Tendências temporais no Brasil

## 💡 Dicas para Análises

- Use **filtros de data** para períodos específicos de análise
- Combine dados **PR + vetores** para análise de risco
- Use **rasters** para modelagem espacial de alta resolução
- Os dados PR são **pontuais** (localização de inquéritos), não casos reportados

## 🔗 Recursos Adicionais

- [Documentação MAP](https://malariaatlas.org/)
- [R Package malariaAtlas](https://github.com/malaria-atlas-project/malariaAtlas)
- [Publicações MAP](https://malariaatlas.org/publications/)